## Home Credit Default Risk — late submission

Runs the end-to-end pipeline from [pinheiro-dataworks/home-credit](https://github.com/pinheiro-dataworks/home-credit) (features → stats → train → predict) on the competition data and writes `submission.csv`.

Optuna trials are reduced from 50 to 20 to fit Kaggle's session time limit.

In [ ]:
import glob, os, shutil, subprocess, sys
os.chdir('/kaggle/working')
if not os.path.exists('home-credit'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/pinheiro-dataworks/home-credit.git'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn>=1.6', 'lightgbm', 'optuna',
                'optuna-integration', 'imbalanced-learn', 'mlflow', 'shap', 'pyarrow', 'pyyaml'], check=True)
import sklearn, lightgbm, optuna, mlflow
print(sys.version, sklearn.__version__, lightgbm.__version__, optuna.__version__, mlflow.__version__)

In [ ]:
REPO = '/kaggle/working/home-credit'
src = os.path.dirname(glob.glob('/kaggle/input/**/application_train.csv', recursive=True)[0])
raw = os.path.join(REPO, 'data', 'raw')
os.makedirs(raw, exist_ok=True)
for f in glob.glob(os.path.join(src, '*.csv')):
    dst = os.path.join(raw, os.path.basename(f))
    if not os.path.exists(dst):
        os.symlink(f, dst)
print(sorted(os.listdir(raw)))
p = os.path.join(REPO, 'params.yaml')
txt = open(p).read().replace('n_trials: 50 ', 'n_trials: 20 ')
open(p, 'w').write(txt)
print([l for l in txt.splitlines() if 'n_trials' in l])

In [ ]:
def run(stage):
    print(f'===== {stage} =====', flush=True)
    proc = subprocess.Popen([sys.executable, 'pipeline.py', '--stage', stage], cwd=REPO,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='', flush=True)
    assert proc.wait() == 0, f'stage {stage} failed'

for stage in ['features', 'stats', 'train', 'predict']:
    run(stage)

In [ ]:
import pandas as pd, json
sub = pd.read_csv(os.path.join(REPO, 'data/processed/submission.csv'))
print(sub.shape, sub['TARGET'].describe())
assert len(sub) == 48744 and sub['SK_ID_CURR'].is_unique and sub['TARGET'].notna().all()
sub.to_csv('/kaggle/working/submission.csv', index=False)
print(json.load(open(os.path.join(REPO, 'models/artifacts/metrics.json'))))
# keep the output small: drop the cloned repo (data/processed parquet, mlruns)
shutil.rmtree(REPO)
print(os.listdir('/kaggle/working'))